# MBEC Experiment 1 — Plain Supervised ResNet50 Baseline

This notebook adds the missing **ordinary supervised ImageNet-transfer baseline** under the exact locked ISIC 2019 protocol used by the paper.

It intentionally does **not** change the proposed final ensemble. Its purpose is to answer the reviewer question:

> Does the proposed self-supervised framework improve over a plain supervised ImageNet-pretrained ResNet50 trained under the same split and imbalance protocol?

### Fair-comparison design
- Exact locked ISIC split: 20,264 / 2,533 / 2,534
- Same 8-class order: MEL, NV, BCC, AK, BKL, DF, VASC, SCC
- Same five seeds as multi-seed SimCLR: 21, 35, 42, 52, 62
- Same 224×224 RGB input and ResNet50 preprocessing
- Same square-root inverse-frequency class weighting used by the paper
- 8-epoch frozen-head warm-up, then up to 35 epochs fine-tuning
- BatchNorm frozen during fine-tuning
- Validation BACC is the checkpoint criterion
- Locked internal test used only after selection
- 5,000-replicate paired stratified bootstrap against the already-frozen proposed final model

## Kaggle Add Input

**Required:** attach either:
1. J1 protocol-lock output containing `original_80_10_10_locked_ids.csv`, **or**
2. the saved `last-part-of-the-previous-one` output containing `rotation_fixed_train_val_test_split.json`.

**Recommended:** also attach the saved `confidence-gated-ensemble` output containing `final_internal_probabilities.npz`; this enables the paired bootstrap against the proposed final model.

The notebook first searches attached inputs for the ISIC images. If they are absent, it downloads the exact same 224×224 archive used by the original thesis notebook. **Turn Kaggle Internet ON** in that case.

In [1]:
# CELL 1 — imports/config
import os, gc, json, random, shutil, zipfile, urllib.request
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
)

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except Exception: pass

SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

KAGGLE_INPUT=Path('/kaggle/input')
ROOT=Path('/kaggle/working/mbec_plain_supervised_resnet50')
MODEL_DIR=ROOT/'models'; TABLE_DIR=ROOT/'tables'; AUDIT_DIR=ROOT/'audit'
for p in [ROOT,MODEL_DIR,TABLE_DIR,AUDIT_DIR]: p.mkdir(parents=True,exist_ok=True)

IMG_SIZE=224; NUM_CLASSES=8
CLASS_NAMES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
BASELINE_SEEDS=[21,35,42,52,62]
HEAD_EPOCHS=8; HEAD_LR=1e-3
FINETUNE_EPOCHS=35; FINETUNE_LR=3e-5
WEIGHT_DECAY=1e-4; DROPOUT=0.30
TRAIN_BATCH=32; EVAL_BATCH=64; PATIENCE=8
TRAIN_BASELINE=True

TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# CELL 2 — discover/load exact locked split

def find_all(filename):
    return sorted(KAGGLE_INPUT.rglob(filename), key=lambda p:(len(p.parts),str(p)))

locked_hits=find_all('original_80_10_10_locked_ids.csv')
split_hits=find_all('rotation_fixed_train_val_test_split.json')

if locked_hits:
    src=locked_hits[0]
    LOCKED_DF=pd.read_csv(src)
    need={'split','image_id','label'}
    if not need.issubset(LOCKED_DF.columns):
        raise ValueError(f'Locked ID CSV missing {need-set(LOCKED_DF.columns)}')
    LOCKED_DF['split']=LOCKED_DF['split'].astype(str).str.upper()
    LOCKED_DF['image_id']=LOCKED_DF['image_id'].astype(str)
    LOCKED_DF['label']=LOCKED_DF['label'].astype(int)
elif split_hits:
    preferred=[p for p in split_hits if 'rotation_ssl_paperlike' in str(p) and 'splits' in str(p)]
    src=preferred[0] if preferred else split_hits[0]
    with open(src,'r') as f: info=json.load(f)
    rows=[]
    for key,split in [('train','TRAIN'),('val','VAL'),('test','TEST')]:
        for r in info[key]:
            rows.append({'split':split,'image_id':Path(str(r['filepath'])).stem,'label':int(r['label'])})
    LOCKED_DF=pd.DataFrame(rows)
else:
    raise FileNotFoundError('Attach J1 output or last-part-of-the-previous-one output.')

expected={'TRAIN':20264,'VAL':2533,'TEST':2534}
counts=LOCKED_DF['split'].value_counts().to_dict()
for s,n in expected.items():
    if int(counts.get(s,-1))!=n: raise RuntimeError(f'{s} size mismatch: {counts.get(s)} != {n}')
if LOCKED_DF['image_id'].duplicated().any(): raise RuntimeError('Duplicate locked image IDs detected.')

LOCKED_DF['class_name']=LOCKED_DF['label'].map({i:x for i,x in enumerate(CLASS_NAMES)})
display(LOCKED_DF.groupby(['split','class_name']).size().unstack(fill_value=0))
LOCKED_DF.to_csv(AUDIT_DIR/'locked_protocol.csv',index=False)
print('Using locked split source:',src)

class_name   AK   BCC   BKL   DF   MEL     NV  SCC  VASC
split                                                   
TEST         86   333   263   24   452   1288   63    25
TRAIN       694  2658  2099  191  3618  10300  502   202
VAL          87   332   262   24   452   1287   63    26

Using locked split source: /kaggle/input/notebooks/anirbanroysudipto/journal-extension-j1/j1_original_protocol_lock/tables/original_80_10_10_locked_ids.csv


In [3]:
# CELL 3 — locate attached ISIC images or download exact thesis archive
needed=set(LOCKED_DF['image_id'].astype(str))
IMAGE_EXTS={'.jpg','.jpeg','.png','.bmp','.webp'}

def scan(root, ids):
    out={}
    if not root.exists(): return out
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS and p.stem in ids and p.stem not in out:
            out[p.stem]=str(p)
    return out

print('Scanning /kaggle/input...')
image_map=scan(KAGGLE_INPUT,needed)
print('Matched attached:',len(image_map),'/',len(needed))

if len(image_map)<len(needed):
    local=Path('/kaggle/working/isic2019_mbec_baseline_data')
    zpath=local/'data.zip'; ex=local/'extracted'; local.mkdir(parents=True,exist_ok=True)
    url='https://huggingface.co/datasets/MKZuziak/ISIC_2019_224/resolve/main/data.zip'
    if not zpath.exists():
        print('Downloading ISIC 2019 224x224...')
        urllib.request.urlretrieve(url,zpath)
    if not ex.exists() or not any(ex.rglob('*.jpg')):
        if ex.exists(): shutil.rmtree(ex)
        ex.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(zpath,'r') as zf: zf.extractall(ex)
    image_map.update(scan(ex,needed))

missing=sorted(needed-set(image_map))
if missing: raise FileNotFoundError(f'{len(missing)} locked images missing. Examples: {missing[:20]}')
LOCKED_DF['filepath']=LOCKED_DF['image_id'].map(image_map)
TRAIN_DF=LOCKED_DF[LOCKED_DF.split=='TRAIN'].reset_index(drop=True)
VAL_DF=LOCKED_DF[LOCKED_DF.split=='VAL'].reset_index(drop=True)
TEST_DF=LOCKED_DF[LOCKED_DF.split=='TEST'].reset_index(drop=True)
print(len(TRAIN_DF),len(VAL_DF),len(TEST_DF))

Scanning /kaggle/input...
Matched attached: 25331 / 25331
20264 2533 2534


In [4]:
# CELL 4 — tf.data + exact class weights + metrics
AUTOTUNE=tf.data.AUTOTUNE

def decode(path,label):
    x=tf.io.read_file(path)
    x=tf.io.decode_image(x,channels=3,expand_animations=False)
    x=tf.image.resize(x,[IMG_SIZE,IMG_SIZE],method=tf.image.ResizeMethod.BILINEAR)
    x=tf.cast(x,tf.float32)/255.0
    x.set_shape([IMG_SIZE,IMG_SIZE,3])
    return x,tf.cast(label,tf.int32)

def aug(x,y):
    x=tf.image.random_flip_left_right(x); x=tf.image.random_flip_up_down(x)
    k=tf.random.uniform([],0,4,dtype=tf.int32); x=tf.image.rot90(x,k)
    return x,y

def make_ds(df,batch,training=False):
    ds=tf.data.Dataset.from_tensor_slices((df.filepath.astype(str).to_numpy(),df.label.astype(np.int32).to_numpy()))
    if training: ds=ds.shuffle(len(df),seed=SEED,reshuffle_each_iteration=True)
    ds=ds.map(decode,num_parallel_calls=AUTOTUNE)
    if training: ds=ds.map(aug,num_parallel_calls=AUTOTUNE)
    return ds.batch(batch).prefetch(AUTOTUNE)

TRAIN_DS=make_ds(TRAIN_DF,TRAIN_BATCH,True); VAL_DS=make_ds(VAL_DF,EVAL_BATCH); TEST_DS=make_ds(TEST_DF,EVAL_BATCH)

counts=TRAIN_DF.label.value_counts().sort_index().reindex(range(NUM_CLASSES))
N=len(TRAIN_DF); raw=np.sqrt(N/counts.to_numpy(dtype=np.float64)); w=.99*raw/raw.max()
CLASS_WEIGHT={i:float(x) for i,x in enumerate(w)}
pd.DataFrame({'class_id':range(8),'class_name':CLASS_NAMES,'train_count':counts.to_numpy(),'weight':w}).to_csv(TABLE_DIR/'class_weights.csv',index=False)
print(CLASS_WEIGHT)

def normp(p,eps=1e-12):
    p=np.asarray(p,dtype=np.float64); p=np.nan_to_num(p,nan=eps,posinf=1,neginf=eps); p=np.clip(p,eps,None); p/=p.sum(1,keepdims=True); return p.astype(np.float32)

def specificity(y,pred,K=8):
    cm=confusion_matrix(y,pred,labels=list(range(K))); total=cm.sum(); vals=[]
    for c in range(K):
        tp=cm[c,c]; fn=cm[c,:].sum()-tp; fp=cm[:,c].sum()-tp; tn=total-tp-fn-fp
        vals.append(tn/(tn+fp) if tn+fp else np.nan)
    return float(np.nanmean(vals))

def evalp(y,p):
    p=normp(p); pred=p.argmax(1)
    try: auc=float(roc_auc_score(y,p,multi_class='ovr',average='macro',labels=list(range(8))))
    except Exception: auc=np.nan
    return dict(accuracy=float(accuracy_score(y,pred)),bacc=float(balanced_accuracy_score(y,pred)),macro_f1=float(f1_score(y,pred,average='macro',zero_division=0)),macro_precision=float(precision_score(y,pred,average='macro',zero_division=0)),macro_recall=float(recall_score(y,pred,average='macro',zero_division=0)),macro_specificity=specificity(y,pred),macro_auc_ovr=auc,pred=pred)

I0000 00:00:1787333010.320029      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


{0: 0.22746657947074206, 1: 0.13481346335923808, 2: 0.26538378869995133, 3: 0.5193643587846517, 4: 0.2986384060536615, 5: 0.9899999999999999, 6: 0.9626672414815942, 7: 0.6106607681071212}


In [5]:
# CELL 5 — model + validation-BACC checkpoint

def preprocess(x): return tf.keras.applications.resnet50.preprocess_input(tf.cast(x,tf.float32)*255.0)

def build_model(seed):
    tf.keras.utils.set_random_seed(seed)
    inp=keras.Input((IMG_SIZE,IMG_SIZE,3),name='image')
    x=layers.Lambda(preprocess,name='resnet50_preprocess')(inp)
    backbone=ResNet50(include_top=False,weights='imagenet',input_shape=(IMG_SIZE,IMG_SIZE,3),pooling='avg')
    x=backbone(x,training=False)
    x=layers.Dropout(DROPOUT,name='dropout')(x)
    out=layers.Dense(NUM_CLASSES,activation='softmax',dtype='float32',name='classifier')(x)
    return keras.Model(inp,out,name=f'supervised_resnet50_seed_{seed}'),backbone

def compile_model(model,lr):
    try: opt=keras.optimizers.AdamW(learning_rate=lr,weight_decay=WEIGHT_DECAY,clipnorm=1.0)
    except Exception: opt=keras.optimizers.Adam(learning_rate=lr,clipnorm=1.0)
    model.compile(opt,loss=keras.losses.SparseCategoricalCrossentropy(),metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')])

class ValBACCSaver(keras.callbacks.Callback):
    def __init__(self,val_ds,path,patience=8):
        super().__init__(); self.val_ds=val_ds; self.path=str(path); self.patience=patience; self.best=(-1,-1,-1); self.bad=0; self.rows=[]
        self.y=np.concatenate([y.numpy() for _,y in val_ds]).astype(np.int32)
    def on_epoch_end(self,epoch,logs=None):
        p=normp(self.model.predict(self.val_ds,verbose=0)); pred=p.argmax(1)
        key=(balanced_accuracy_score(self.y,pred),f1_score(self.y,pred,average='macro',zero_division=0),accuracy_score(self.y,pred))
        self.rows.append({'epoch':epoch+1,'val_bacc':key[0],'val_macro_f1':key[1],'val_accuracy':key[2]})
        print(f' val_bacc={key[0]:.6f} macro_f1={key[1]:.6f}')
        if key>self.best:
            self.best=key; self.bad=0; self.model.save_weights(self.path); print(' saved best BACC')
        else: self.bad+=1
        if self.bad>=self.patience: self.model.stop_training=True

def predict(model,ds):
    y=np.concatenate([y.numpy() for _,y in ds]).astype(np.int32)
    return y,normp(model.predict(ds,verbose=0))

In [6]:
# CELL 6 — train/reuse five seeds
rows=[]; val_probs=[]; test_probs=[]; seed_paths=OrderedDict()
for seed in BASELINE_SEEDS:
    sd=MODEL_DIR/f'seed_{seed}'; sd.mkdir(exist_ok=True); best=sd/'best_by_val_bacc.weights.h5'; seed_paths[seed]=best
    tf.keras.backend.clear_session(); gc.collect(); model,backbone=build_model(seed)
    if TRAIN_BASELINE or not best.exists():
        print('\n=== seed',seed,'head warm-up ===')
        backbone.trainable=False; compile_model(model,HEAD_LR)
        model.fit(TRAIN_DS,validation_data=VAL_DS,epochs=HEAD_EPOCHS,class_weight=CLASS_WEIGHT,verbose=1)
        print('=== seed',seed,'fine-tuning ===')
        backbone.trainable=True
        for layer in backbone.layers:
            if isinstance(layer,layers.BatchNormalization): layer.trainable=False
        compile_model(model,FINETUNE_LR)
        saver=ValBACCSaver(VAL_DS,best,PATIENCE)
        reduce=keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=.5,patience=3,min_lr=1e-7,verbose=1)
        model.fit(TRAIN_DS,validation_data=VAL_DS,epochs=FINETUNE_EPOCHS,class_weight=CLASS_WEIGHT,callbacks=[saver,reduce],verbose=1)
        pd.DataFrame(saver.rows).to_csv(sd/'validation_bacc_history.csv',index=False)
    model.load_weights(best)
    yv,pv=predict(model,VAL_DS); yt,pt=predict(model,TEST_DS)
    if not np.array_equal(yv,VAL_DF.label.to_numpy(np.int32)) or not np.array_equal(yt,TEST_DF.label.to_numpy(np.int32)): raise RuntimeError('Order mismatch')
    rv=evalp(yv,pv); rt=evalp(yt,pt); val_probs.append(pv); test_probs.append(pt)
    rows.append({'seed':seed,'val_accuracy':rv['accuracy'],'val_bacc':rv['bacc'],'val_macro_f1':rv['macro_f1'],'test_accuracy':rt['accuracy'],'test_bacc':rt['bacc'],'test_macro_f1':rt['macro_f1'],'test_macro_specificity':rt['macro_specificity'],'test_macro_auc_ovr':rt['macro_auc_ovr'],'weights':str(best)})
    del model,backbone; gc.collect()

SEED_RESULTS=pd.DataFrame(rows); display(SEED_RESULTS); SEED_RESULTS.to_csv(TABLE_DIR/'seed_results.csv',index=False)
Y_VAL=VAL_DF.label.to_numpy(np.int32); Y_TEST=TEST_DF.label.to_numpy(np.int32)
VAL_PROB=normp(np.mean(np.stack(val_probs),axis=0)); TEST_PROB=normp(np.mean(np.stack(test_probs),axis=0))
VAL_RESULT=evalp(Y_VAL,VAL_PROB); TEST_RESULT=evalp(Y_TEST,TEST_PROB)
print('5-seed baseline:',{k:v for k,v in TEST_RESULT.items() if k!='pred'})
np.savez_compressed(ROOT/'supervised_resnet50_5seed_probabilities.npz',y_val=Y_VAL,y_test=Y_TEST,val_prob=VAL_PROB,test_prob=TEST_PROB,seeds=np.array(BASELINE_SEEDS))

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

=== seed 21 head warm-up ===
Epoch 1/8
  3/634 ━━━━━━━━━━━━━━━━━━━━ 27s 44ms/step - accuracy: 0.0694 - loss: 0.6788       

I0000 00:00:1787333028.090136      65 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


634/634 ━━━━━━━━━━━━━━━━━━━━ 57s 71ms/step - accuracy: 0.5690 - loss: 0.3453 - val_accuracy: 0.6317 - val_loss: 1.0262
Epoch 2/8
634/634 ━━━━━━━━━━━━━━━━━━━━ 31s 49ms/step - accuracy: 0.6163 - loss: 0.2923 - val_accuracy: 0.6561 - val_loss: 0.9393
Epoch 3/8
634/634 ━━━━━━━━━━━━━━━━━━━━ 31s 49ms/step - accuracy: 0.6340 - loss: 0.2789 - val_accuracy: 0.6577 - val_loss: 0.9425
Epoch 4/8
634/634 ━━━━━━━━━━━━━━━━━━━━ 31s 49ms/step - accuracy: 0.6333 - loss: 0.2725 - val_accuracy: 0.6368 - val_loss: 0.9685
Epoch 5/8
634/634 ━━━━━━━━━━━━━━━━━━━━ 31s 49ms/step - accuracy: 0.6398 - loss: 0.2641 - val_accuracy: 0.6873 - val_loss: 0.8657
Epoch 6/8
634/634 ━━━━━━━━━━━━━━━━━━━━ 31s 49ms/step - accuracy: 0.6426 - loss: 0.2631 - val_accuracy: 0.6565 - val_loss: 0.9189
Epoch 7/8
634/634 ━━━━━━━━━━━━━━━━━━━━ 31s 48ms/step - accuracy: 0.6387 - loss: 0.2633 - val_accuracy: 0.6794 - val_loss: 0.8796
Epoch 8/8
634/634 ━━━━━━━━━━━━━━━━━━━━ 31s 49ms/step - accuracy: 0.6432 - loss: 0.2621 - val_accuracy: 0.69

   seed  val_accuracy  val_bacc  val_macro_f1  test_accuracy  test_bacc  \
0    21      0.835373  0.772196      0.775055       0.830308   0.759727   
1    35      0.846427  0.775152      0.786111       0.833465   0.775973   
2    42      0.839716  0.776456      0.785739       0.836622   0.786606   
3    52      0.816028  0.774142      0.770857       0.817285   0.785408   
4    62      0.852349  0.786811      0.796975       0.838990   0.781593   

   test_macro_f1  test_macro_specificity  test_macro_auc_ovr  \
0       0.772559                0.968857            0.969987   
1       0.782050                0.970052            0.967097   
2       0.786907                0.970998            0.968798   
3       0.786507                0.968632            0.966778   
4       0.793389                0.969932            0.971687   

                                             weights  
0  /kaggle/working/mbec_plain_supervised_resnet50...  
1  /kaggle/working/mbec_plain_supervised_resnet50...  

5-seed baseline: {'accuracy': 0.8484609313338595, 'bacc': 0.8016992291101837, 'macro_f1': 0.8147701647207382, 'macro_precision': 0.8357794678612167, 'macro_recall': 0.8016992291101837, 'macro_specificity': 0.9720363220551693, 'macro_auc_ovr': 0.9759547674116642}


In [7]:
# CELL 7 — report files + paired bootstrap vs frozen final proposed model
pd.DataFrame(confusion_matrix(Y_TEST,TEST_RESULT['pred'],labels=range(8)),index=[f'true_{x}' for x in CLASS_NAMES],columns=[f'pred_{x}' for x in CLASS_NAMES]).to_csv(TABLE_DIR/'baseline_test_confusion_matrix.csv')
pd.DataFrame(classification_report(Y_TEST,TEST_RESULT['pred'],target_names=CLASS_NAMES,output_dict=True,zero_division=0)).T.to_csv(TABLE_DIR/'baseline_test_classification_report.csv')

comparison=[{'model':'Supervised ImageNet ResNet50 (5-seed mean)',**{k:TEST_RESULT[k] for k in ['accuracy','bacc','macro_f1','macro_specificity','macro_auc_ovr']}}]
final_hits=find_all('final_internal_probabilities.npz'); FINAL_PROB=None
if final_hits:
    preferred=[p for p in final_hits if 'reuse_only_all_model_ensemble' in str(p) or 'j1_original_protocol_lock' in str(p)]
    fp=preferred[0] if preferred else final_hits[0]
    with np.load(fp) as z:
        yf=np.asarray(z['y_test'],dtype=np.int32)
        key='final_test_prob' if 'final_test_prob' in z else 'stacking_bias_test_prob'
        FINAL_PROB=normp(z[key])
    if not np.array_equal(yf,Y_TEST): raise RuntimeError('Frozen final y_test does not align.')
    fr=evalp(Y_TEST,FINAL_PROB)
    comparison.append({'model':'Proposed final stacking + class bias',**{k:fr[k] for k in ['accuracy','bacc','macro_f1','macro_specificity','macro_auc_ovr']}})

COMP=pd.DataFrame(comparison); display(COMP); COMP.to_csv(TABLE_DIR/'journal_baseline_comparison.csv',index=False)

if FINAL_PROB is not None:
    def strat_idx(y,rng):
        blocks=[]
        for c in np.unique(y):
            ii=np.where(y==c)[0]; blocks.append(rng.choice(ii,len(ii),replace=True))
        out=np.concatenate(blocks); rng.shuffle(out); return out
    rng=np.random.default_rng(42); fpred=FINAL_PROB.argmax(1); bpred=TEST_PROB.argmax(1); diffs=[]
    observed=balanced_accuracy_score(Y_TEST,fpred)-balanced_accuracy_score(Y_TEST,bpred)
    for _ in range(5000):
        ii=strat_idx(Y_TEST,rng); diffs.append(balanced_accuracy_score(Y_TEST[ii],fpred[ii])-balanced_accuracy_score(Y_TEST[ii],bpred[ii]))
    diffs=np.asarray(diffs); lo,hi=np.quantile(diffs,[.025,.975]); p=2*min(np.mean(diffs<=0),np.mean(diffs>=0))
    BOOT=pd.DataFrame([{'comparison':'Final Proposed - Supervised ResNet50','observed_bacc_difference':observed,'ci95_low':lo,'ci95_high':hi,'prob_final_greater':np.mean(diffs>0),'two_sided_bootstrap_p':min(float(p),1.0),'replicates':5000,'seed':42}])
    display(BOOT); BOOT.to_csv(TABLE_DIR/'bootstrap_final_vs_supervised_resnet50.csv',index=False)

print('Outputs:',ROOT)

                                        model  accuracy      bacc  macro_f1  \
0  Supervised ImageNet ResNet50 (5-seed mean)  0.848461  0.801699  0.814770   
1        Proposed final stacking + class bias  0.838990  0.839549  0.797961   

   macro_specificity  macro_auc_ovr  
0           0.972036       0.975955  
1           0.972839       0.976656  

                             comparison  observed_bacc_difference  ci95_low  \
0  Final Proposed - Supervised ResNet50                   0.03785  0.011409   

   ci95_high  prob_final_greater  two_sided_bootstrap_p  replicates  seed  
0   0.065394              0.9964                 0.0072        5000    42  

Outputs: /kaggle/working/mbec_plain_supervised_resnet50


### Paper use
Use `tables/journal_baseline_comparison.csv` to add the supervised baseline to the main internal-performance table. If the frozen final probabilities were attached, also report `tables/bootstrap_final_vs_supervised_resnet50.csv`.

This baseline should be described as **ImageNet-supervised ResNet50 transfer learning**, not as an SSL method.